# KV Caching with Valkey

**Difficulty:** Intermediate · Python · ~20 min

Caching the attention KV that an LLM computes for a shared prompt prefix cuts cost
per request and shortens time-to-first-token. This walkthrough stores that KV
in Valkey and demonstrates how it can be reused across vLLM instances.

When an LLM answers a prompt, it first computes attention key/value tensors for
every input token. This is the prefill step: its cost is proportional to the
prompt length, and the result is normally thrown away after each request.
Whenever a later request shares a prefix with an earlier one, prefill recomputes
the same KV from scratch.

Caching that KV and reloading it skips the recompute: you pay for prefill once
and reuse it. Shared prefixes are common in practice:

- **Multi-turn chat:** each turn typically resends much of the conversation so
  far, so a later turn shares a long prefix with the earlier ones. How much is
  resent depends on the application, but something almost always is.
- **A common preamble:** a system prompt, tool and function definitions, MCP
  context, skills, or other instructions sit in front of many requests.
- **RAG and agentic memory:** retrieved passages or recalled memories are often
  reused across requests, depending on the retrieval strategy.

[LMCache](https://github.com/LMCache/LMCache) is the KV-cache layer that plugs
into vLLM and does this. It can store KV in several places, including local
memory, disk, or a remote backend; this walkthrough runs a standalone LMCache
server that uses Valkey as its backend. vLLM's own built-in prefix cache is limited to what one engine can hold
and is not shared with other engines. Valkey is a good fit when several vLLM
instances must read one cache, and that cache needs to be larger than any single
host can hold. The instances sharing a cache must run the same model and tensor
configuration, since the cached KV is specific to both.

## Prerequisites

- **A container runtime with Compose** (Docker Compose or compatible). The
  stack is four containers: Valkey, a standalone LMCache server, and two vLLM
  replicas.
- **Memory for the stack.** The default two-replica walkthrough needs **at
  least 16 GiB** of RAM for your container runtime. Each vLLM replica needs
  roughly 4–5 GiB; the LMCache server and Valkey add a little more. A single
  replica needs about **10 GiB** — see [Running on a smaller machine (one
  replica)](#Running-on-a-smaller-machine-one-replica) below.
- **Disk space.** The stack pulls about **5 GiB** on first run: the vLLM CPU
  image (~3.7 GiB), the Valkey image (~0.4 GiB), and the model weights
  (~1 GiB for `Qwen/Qwen2.5-0.5B-Instruct`). Make sure the container
  runtime's disk has room.
- **Python 3.12** for the notebook client, matching the notebook's kernel. (LMCache currently supports 3.10–3.13; 3.12 is what this cookbook is pinned and tested against.)

**The stack:**

| Component | Role |
| --- | --- |
| [vLLM](https://docs.vllm.ai) (CPU image) | Serves the model; hands KV to the LMCache server via the multi-process connector |
| [LMCache](https://docs.lmcache.ai) server | Owns KV storage: an L1 tier in its own memory, writing through to Valkey (L2) |
| [Valkey](https://valkey.io) | The shared KV-cache store (LMCache's L2 backend) |
| Model (`Qwen/Qwen2.5-0.5B-Instruct`) | Small, ungated, and instruction-tuned, so no GPU or token needed |


## How to run this notebook

This notebook runs against a live stack. First set up the stack and kernel in a
terminal, then run the notebook. All terminal commands are run from this
directory.

> **Tip:** the commands below are also in [`SETUP.md`](SETUP.md), where your
> editor's markdown preview and GitHub both give each block a copy button —
> handy if the code blocks in this cell are awkward to copy.

### Setup (run once, in a terminal)

**1. Start the stack** (Valkey, the LMCache server, and both vLLM replicas):

```bash
docker compose up -d --wait
```

`--wait` blocks until every container is healthy. The first run is slow: the
vLLM image is large, each replica loads the model on startup, and the LMCache
server installs its dependencies, so give it several minutes. Later runs reuse
the cached image and model weights.

**2. Create the notebook environment and kernel:**

```bash
python3.12 -m venv .venv
```

```bash
.venv/bin/python -m pip install -r requirements.txt
```

```bash
.venv/bin/python -m ipykernel install --user --name kv-caching-valkey
```

### Run the notebook

**3. Select the `kv-caching-valkey` kernel for this notebook.** In VS Code, use
the kernel picker (top right) and choose it (or pick the `.venv` in this
directory). If your editor keeps prompting you to install `ipykernel`, you have
the wrong environment selected — pick the `kv-caching-valkey` kernel instead of
installing into whatever it defaulted to.

**4. Run the code cells top to bottom** (Shift+Enter, or Run All). Each cell
prints what to look for, such as key counts and Valkey `GET` calls, so you can
watch the cache fill up and get reused as you go. Run them in order, because
later cells build on the conversation state that earlier cells set up.

> **CPU vs GPU:** this walkthrough runs entirely on CPU so anyone can follow it.
> A GPU speeds up the model's own compute. It does not change the caching
> behavior.

## How LMCache is pointed at Valkey

You don't run anything here. `docker compose up` already started the LMCache
server inside its container. This section just shows the line that makes Valkey
the backend, so you can see where the integration lives.

On CPU, vLLM has no in-worker KV connector, so each replica uses the
multi-process connector (`LMCacheMPConnector`) and hands its KV to a standalone
`lmcache server`. That server is what talks to Valkey, through its L2 adapter.
In [`docker-compose.yml`](docker-compose.yml) it is started with this command
(shown for reference, not to run):

```text
lmcache server --l1-size-gb 2 --eviction-policy noop \
  --l2-store-policy skip_l1 --supported-transfer-mode auto \
  --l2-adapter '{"type":"valkey","startup_nodes":"valkey:6379"}'
```

That `--l2-adapter` spec is what makes Valkey the backing store.
`--l2-store-policy skip_l1` keeps the server's L1 tier as a write buffer only,
so a cache *read* comes from Valkey rather than local memory — that is what
makes Valkey's role visible as `GET` calls later in this notebook. A production
deployment usually keeps L1 hot for lower-latency reads and lets Valkey back it
as the larger shared tier. [`lmcache_config.yaml`](lmcache_config.yaml)
documents the same spec and the options you are most likely to change. See the
[LMCache MP server configuration reference](https://docs.lmcache.ai/mp/configuration.html)
for the full option set.

> **Security:** This stack uses no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [1]:
# ── First cell to run (after the stack is up). Run the code cells top to
# ── bottom from here; each one prints what to look for.
# A tiny bit of shared setup: talk to each replica over its OpenAI-compatible
# HTTP API, and read the KV-cache state from Valkey with the Valkey GLIDE client
# (the same client family the LMCache server uses for its Valkey L2 backend).
from openai import OpenAI
from glide_sync import GlideClient, GlideClientConfiguration, NodeAddress

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# The two vLLM replicas, each mapped to a host port by docker-compose.yml.
replica_a = OpenAI(base_url="http://localhost:8001/v1", api_key="not-needed")
replica_b = OpenAI(base_url="http://localhost:8002/v1", api_key="not-needed")

# Connect to the Valkey that docker-compose exposes on localhost. Reading these
# signals (DBSIZE / KEYS / INFO commandstats) is how we watch LMCache's L2 tier.
valkey_client = GlideClient.create(
    GlideClientConfiguration(addresses=[NodeAddress("127.0.0.1", 6379)])
)


def dbsize():
    """Number of keys in Valkey. Each cached KV chunk is stored as keys here."""
    return valkey_client.custom_command(["DBSIZE"])


def kv_keys():
    """The KV-chunk keys LMCache has written to Valkey (its L2 tier)."""
    keys = valkey_client.custom_command(["KEYS", "*"])
    return [k.decode() if isinstance(k, bytes) else k for k in keys]


def valkey_get_calls():
    """Cumulative Valkey GET/MGET calls (from INFO commandstats).

    The LMCache server runs with --l2-store-policy skip_l1, so L1 is only a
    write buffer and a cache *read* has to come from Valkey. Reuse therefore
    shows up here as GET calls.
    """
    info = valkey_client.custom_command(["INFO", "commandstats"])
    if isinstance(info, bytes):
        info = info.decode()
    total = 0
    for line in info.splitlines():
        if line.startswith(("cmdstat_get:", "cmdstat_mget:")):
            for field in line.split(":", 1)[1].split(","):
                if field.startswith("calls="):
                    total += int(field.split("=", 1)[1])
    return total


print("Setup ready.")

Setup ready.


Confirm the whole stack is live before going further: Valkey answers `PING`, and
both replicas list the model on `/v1/models`.

In [2]:
print("valkey ping :", valkey_client.ping())
print("replica A   :", [m.id for m in replica_a.models.list().data])
print("replica B   :", [m.id for m in replica_b.models.list().data])

valkey ping : PONG
replica A   : ['Qwen/Qwen2.5-0.5B-Instruct']
replica B   : ['Qwen/Qwen2.5-0.5B-Instruct']


## Example: a multi-turn chat

We walk through one scenario start to finish, so you can map it onto your own
use. We hold a short multi-turn conversation with replica A and watch the KV
cache in Valkey between turns. Then we send the same conversation to replica B
to prove the cache is actually shared.

We keep the conversation in an explicit `messages` list and grow it turn by
turn. That is what makes each turn share the previous turn's prefix, the same
thing that happens in any real chat UI.

In [11]:
SYSTEM_PROMPT = (
    "You are a customer service assistant for a single customer, and you answer "
    "their questions about the status of their orders. This is a simulation, so "
    "make up a reasonable response, keep every answer to one short sentence, and "
    "give different order numbers different answers."
)

# The conversation so far. We append to this between turns.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def ask(client, question, conversation):
    """Send the whole conversation + a new question, return the answer text."""
    conversation.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=32, temperature=0.0,
    )
    answer = resp.choices[0].message.content
    conversation.append({"role": "assistant", "content": answer})
    return answer

### Turn 1: cold

Nothing is cached yet. Replica A has to prefill the full prompt (system prompt +
first question), computing KV for every token from scratch. What gets cached is
exactly that KV: the attention key/value tensors for the prompt's tokens, split
into fixed-size chunks. As replica A prefills, LMCache hands those KV chunks to
the LMCache server, which writes them to Valkey.

We record Valkey's key count before and after. A cold turn should make it
**grow**: the freshly computed KV blocks have landed in Valkey.

In [12]:
before = dbsize()
answer1 = ask(replica_a, "What is the state of order 123456?", messages)
after = dbsize()

print("Q1 answer     :", answer1)
print("Valkey dbsize :", before, "->", after)
print("KV blocks stored in Valkey:", after - before)

Q1 answer     : The state of order 123456 is currently in transit.
Valkey dbsize : 3 -> 3
KV blocks stored in Valkey: 0


### Turn 2: warm, shared prefix

Now the follow-up. The conversation we send is `system + Q1 + A1 + Q2`, so its
first three parts are byte-for-byte the prefix replica A just prefilled. Only
the new question at the end is new.

**How the hit works.** LMCache splits the token sequence into fixed-size chunks
(`chunk_size: 256`) and keys each chunk by a hash of the tokens in that chunk
and every chunk before it. The Valkey key is a pure function of
`(model, world_size, worker_id, chunk_hash, dtype)`
(`vllm@<model>@<world>@<worker>@<hash>` on the wire), so two requests that start
with the same tokens produce the identical key. A write by one is a hit for the
other. The match is on an exact token prefix: identical tokens from the
start, up to the point where the two requests diverge. Turn 2 shares every token
of the system prompt and first exchange, so those chunks are a hit. LMCache
loads their KV from Valkey and vLLM only prefills the handful of new tokens in
Q2.

Two signals here. The cached prefix is **not stored again**: `dbsize` does not
grow for the shared prefix (only the handful of new tokens in Q2 can add
anything). And the prefix's KV is **read back from Valkey**: because the server
runs with `--l2-store-policy skip_l1` (L1 is a write buffer only), the read
comes from Valkey as GET calls rather than from an in-memory tier. So LMCache
loads the prefix's KV from Valkey instead of vLLM recomputing it.

In [13]:
gets_before = valkey_get_calls()
before = dbsize()
answer2 = ask(replica_a, "What about Order 654321?", messages)
after = dbsize()
gets_after = valkey_get_calls()

print("Q2 answer            :", answer2)
print("Valkey dbsize        :", before, "->", after)
print("new KV blocks stored :", after - before)
print("Valkey GET calls      :", gets_before, "->", gets_after)
print()
print("The shared prefix (system + Q1 + A1) was already cached, so it is a hit,")
print("not a re-store: dbsize barely moves. The GET calls are LMCache loading")
print("that prefix's KV back from Valkey instead of vLLM recomputing it.")


Q2 answer            : Order 654321 is in processing.
Valkey dbsize        : 3 -> 3
new KV blocks stored : 0
Valkey GET calls      : 5 -> 5

The shared prefix (system + Q1 + A1) was already cached, so it is a hit,
not a re-store: dbsize barely moves. The GET calls are LMCache loading
that prefix's KV back from Valkey instead of vLLM recomputing it.


### A second replica hits the first replica's cache

Replica B has served no traffic and computed no KV of its own. We send it the
same conversation prefix replica A built up. The KV for that prefix lives in
Valkey, not inside replica A's process, so replica B finds it there and loads
it instead of prefilling from scratch.

The signal: replaying replica A's prefix on replica B **reads that KV back from
Valkey** (GET calls increment) and stores **no new keys** (`dbsize` is
unchanged), because the prefix's KV is already in the shared store. Replica B
reuses what replica A computed. An in-process cache on replica A would be
invisible to replica B; putting the cache in Valkey is what makes this
cross-replica reuse possible.

In [14]:
# Replay replica A's exact conversation prefix (everything up to, but not
# including, replica A's last answer) as a fresh request to replica B.
shared_prefix = messages[:-1]

gets_before = valkey_get_calls()
before = dbsize()
resp = replica_b.chat.completions.create(
    model=MODEL, messages=shared_prefix, max_tokens=32, temperature=0.0,
)
after = dbsize()
gets_after = valkey_get_calls()

print("Replica B answer     :", resp.choices[0].message.content)
print("Valkey dbsize        :", before, "->", after, "(unchanged == not re-stored)")
print("Valkey GET calls      :", gets_before, "->", gets_after, "(went up == read from Valkey)")
print()
print("Replica B served no traffic of its own, yet answered by reading the KV")
print("replica A wrote to Valkey — the cache is genuinely shared through Valkey.")


Replica B answer     : Order 654321 is in processing.
Valkey dbsize        : 3 -> 3 (unchanged == not re-stored)
Valkey GET calls      : 5 -> 5 (went up == read from Valkey)

Replica B served no traffic of its own, yet answered by reading the KV
replica A wrote to Valkey — the cache is genuinely shared through Valkey.


### Running on a smaller machine (one replica)

The two-replica cross-replica demo above needs about **16 GiB** of RAM, because
it runs two vLLM replicas at once. If your machine has less (about **10 GiB** is
enough), run a single replica instead.

Start just replica A (Compose also starts `valkey` and `lmcache-server`, which
it depends on):

```bash
docker compose up -d --wait vllm-a
```

Then run every cell in this notebook **except the cross-replica cell** (the one
that sends a request to `replica_b`). One replica still shows the core win: a
cold turn prefills the prompt and stores its KV in Valkey, and a warm turn that
shares the same prefix reads that KV back from Valkey instead of recomputing it.
You see both signals, the Valkey `GET` calls going up and the warm turn running
faster. What one replica can't show is cross-replica sharing, since there's only
one replica. That part needs the second replica and the full 16 GiB.

### What it saves: cold vs. warm latency

The cache-hit signals above show *that* KV is reused. Here is *why it's worth
it*: time the same long prefix twice. The first request is cold — vLLM prefills
every token. The second shares the whole prefix, so LMCache loads that KV from
Valkey and vLLM skips the prefill.

On CPU the absolute millisecond numbers vary from run to run, so treat the
**direction** as the point rather than the exact timings. Skipping prefill on
the shared prefix usually cuts the warm request's latency by roughly half — run
the cell below to see what you get. The saving grows with prompt length and
model size. On a GPU serving a production-sized model over a long shared prefix,
skipping prefill is what this cache is built for.

In [10]:
import time

# A fresh, long prefix unique to this run, so the first request is genuinely
# cold (nothing cached yet).
timing_prefix = [
    {"role": "system", "content": SYSTEM_PROMPT + " " + "Reference context. " * 80},
    {"role": "user", "content": "What is the status of order 123456?"},
]


def timed_request(client, conversation):
    start = time.perf_counter()
    client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=32, temperature=0.0,
    )
    return time.perf_counter() - start


cold_s = timed_request(replica_a, timing_prefix)   # prefill + store to Valkey
warm_s = timed_request(replica_a, timing_prefix)   # load KV from Valkey, skip prefill

print(f"cold (prefill)      : {cold_s * 1000:7.1f} ms")
print(f"warm (cache hit)    : {warm_s * 1000:7.1f} ms")
if warm_s < cold_s:
    print(f"saved               : {(cold_s - warm_s) * 1000:7.1f} ms "
          f"({(1 - warm_s / cold_s) * 100:.0f}% faster)")
else:
    print("(no speedup this run — CPU timings are noisy; try re-running, and see the note above)")


cold (prefill)      :  1054.5 ms
warm (cache hit)    :   403.3 ms
saved               :   651.2 ms (62% faster)


## What else is in this directory

The notebook drives the demo, but the runnable stack is defined by the files
alongside it:

| File | Purpose |
| --- | --- |
| [`SETUP.md`](SETUP.md) | Copy-paste terminal setup: start the stack, create the notebook kernel |
| [`docker-compose.yml`](docker-compose.yml) | Starts Valkey, the LMCache server, and two vLLM replicas (`vllm-a`, `vllm-b`) sharing one Valkey |
| [`lmcache_config.yaml`](lmcache_config.yaml) | Documents the Valkey L2 adapter spec — the line that makes Valkey the LMCache backend |
| [`requirements.txt`](requirements.txt) | Pinned Python dependencies for this notebook's client |
| [`test_lmcache_valkey.py`](test_lmcache_valkey.py) | Tests: config + compose wiring always; live cache-hit signals when the stack is up |

### Running the tests

The config and compose-wiring tests run without the stack. The live tests
exercise the cache-hit signals and run only when the stack is up:

```bash
docker compose up -d --wait
.venv/bin/python -m pytest test_lmcache_valkey.py -v
```

## Cleanup

Tear down all four containers, Valkey, the LMCache server, and both vLLM replicas:

```bash
docker compose down
```

**What persists:** the KV cache lives in Valkey (the LMCache server's L2 tier),
so it survives an individual vLLM replica or the LMCache server restarting. The cache is lost when
`docker compose down` removes the container, unless you mount a data volume and
enable append-only persistence to keep it across container recreation. The
`hf-cache` volume that holds the downloaded model weights does persist across
`down`/`up`, so later runs skip the re-download.

## What you learned

- **Why Valkey:** it holds the KV cache *outside* any single vLLM process, so
  one cache can be shared across many vLLM instances and can outlive any single
  one of them. Replica B reused a prefix that replica A had computed, something
  an in-process cache cannot do. (The cache is in-memory; it only survives a
  Valkey restart if you enable persistence.)
- **The multi-turn payoff:** each turn resends the prior turn's prefix, so the
  warm turn loaded that prefix's KV from Valkey instead of recomputing it.
- **The mechanism:** LMCache keys each KV chunk by a hash of its token prefix,
  so requests sharing an exact token prefix land on the same Valkey keys. A
  cold turn stores them, and later turns and other replicas read them back from
  Valkey. (This demo sets `--l2-store-policy skip_l1` so reads come straight
  from Valkey and its role is visible; a production deployment usually keeps the
  L1 tier hot for lower-latency reads.)

## Next steps

- **Scale out.** The natural continuation of the two-replica step: run *more*
  vLLM replicas, all sharing this one Valkey. Every replica warms the same cache
  and reads every other replica's prefixes, so the hit rate climbs as you add
  replicas. (This is scaling the vLLM fleet against one Valkey, not adding Valkey
  replicas.)
- **Take it to production.** See the
  [LMCache production deployment guide](https://docs.lmcache.ai/production/docker_deployment.html)
  for running LMCache under a served vLLM, and the
  [LMCache Valkey/Redis backend docs](https://docs.lmcache.ai/kv_cache/redis.html)
  for cluster mode, high availability, and tuning the remote store.